In [4]:
import pandas as pd

df = pd.read_csv('ref/sup_nature.txt',sep =';' )
df

,NAT_ID,NAT_LB_NOM
0,0,Sans nature
1,40,Sémaphore
2,41,Phare
3,4,Château d'eau - réservoir
4,38,Immeuble
5,39,Local technique
6,42,Mât
7,8,Intérieur galerie
8,9,Intérieur sous-terrain
9,10,Tunnel


In [1]:
import pandas as pd

def lire(chemin):
    return pd.read_csv(chemin, sep=";", encoding="latin-1", dtype=str, low_memory=False)

support  = lire("data/sup_support.txt")
station  = lire("data/sup_station.txt")
antenne  = lire("data/sup_antenne.txt")
bande    = lire("data/sup_bande.txt")

for nom, df in [("support", support), ("station", station), ("antenne", antenne), ("bande", bande)]:
    print(nom, list(df.columns))

support ['SUP_ID', 'STA_NM_ANFR', 'NAT_ID', 'COR_NB_DG_LAT', 'COR_NB_MN_LAT', 'COR_NB_SC_LAT', 'COR_CD_NS_LAT', 'COR_NB_DG_LON', 'COR_NB_MN_LON', 'COR_NB_SC_LON', 'COR_CD_EW_LON', 'SUP_NM_HAUT', 'TPO_ID', 'ADR_LB_LIEU', 'ADR_LB_ADD1', 'ADR_LB_ADD2', 'ADR_LB_ADD3', 'ADR_NM_CP', 'COM_CD_INSEE']
station ['STA_NM_ANFR', 'ADM_ID', 'DEM_NM_COMSIS', 'DTE_IMPLANTATION', 'DTE_MODIF', 'DTE_EN_SERVICE', 'STA_NM_USER']
antenne ['STA_NM_ANFR', 'AER_ID', 'TAE_ID', 'AER_NB_DIMENSION', 'AER_FG_RAYON', 'AER_NB_AZIMUT', 'AER_NB_ALT_BAS', 'SUP_ID']
bande ['STA_NM_ANFR', 'BAN_ID', 'EMR_ID', 'BAN_NB_F_DEB', 'BAN_NB_F_FIN', 'BAN_FG_UNITE']


In [11]:
def infos_antenne(aer_id):
    # 1. L'antenne : AER_ID + TAE_ID + sa station (STA_NM_ANFR)
    a = antenne[antenne["AER_ID"] == str(aer_id)]
    if a.empty:
        return "Antenne introuvable"

    # 2. La station : STA_NM_ANFR + ADM_ID (exploitant) + SUP_ID (support)
    r = a.merge(station, on="STA_NM_ANFR", how="left", suffixes=("", "_sta"))

    # 3. Le support : SUP_ID + NAT_ID + ADR_LB_LIEU + ADR_NM_CP
    r = r.merge(support, on="SUP_ID", how="left", suffixes=("", "_sup"))

    # 4. La bande : BAN_ID + EMR_ID (liée à la station)
    r = r.merge(bande, on="STA_NM_ANFR", how="left", suffixes=("", "_ban"))

    colonnes = ["SUP_ID", "NAT_ID", "ADR_LB_LIEU", "ADR_NM_CP",
                "STA_NM_ANFR", "ADM_ID", "AER_ID", "TAE_ID",
                "BAN_ID", "EMR_ID"]
    return r[[c for c in colonnes if c in r.columns]]

infos_antenne("1400327")

,SUP_ID,NAT_ID,ADR_LB_LIEU,ADR_NM_CP,STA_NM_ANFR,ADM_ID,AER_ID,TAE_ID,BAN_ID,EMR_ID
0,602525,17,NaN,01500,0010030350,2,1400327,18,3099819,1488240
1,602525,17,NaN,01500,0010030350,2,1400327,18,3099821,1488240


In [6]:
def localisation_antenne(aer_id):
    r = antenne[antenne["AER_ID"] == str(aer_id)]
    r = r.merge(station, on="STA_NM_ANFR", how="left", suffixes=("", "_sta"))
    r = r.merge(support, on="SUP_ID", how="left", suffixes=("", "_sup"))
    return r[["AER_ID", "ADR_LB_LIEU", "ADR_NM_CP"]].drop_duplicates()

localisation_antenne("2151551")   #AER_ID

,AER_ID,ADR_LB_LIEU,ADR_NM_CP
0,2151551,NaN,01300


In [15]:
def antennes_du_cp(cp):
    r = antenne.merge(station, on="STA_NM_ANFR", how="left", suffixes=("", "_sta"))
    r = r.merge(support, on="SUP_ID", how="left", suffixes=("", "_sup"))
    r = r[r["ADR_NM_CP"] == str(cp)]
    return r[["AER_ID", "ADR_LB_LIEU", "ADR_NM_CP"]].drop_duplicates()

antennes_du_cp("95750")   # exemple : Chars

,AER_ID,ADR_LB_LIEU,ADR_NM_CP
2082396,10771859,NaN,95750
2082750,6187782,NaN,95750
2082751,6187784,NaN,95750
2086017,8608864,VIS A VIS LA MARNIERE,95750
2086021,8608866,VIS A VIS LA MARNIERE,95750
2086025,8608868,VIS A VIS LA MARNIERE,95750
2086029,9436159,VIS A VIS LA MARNIERE,95750
2086523,10074901,LA CORNOUILLE,95750
2086527,10074903,LA CORNOUILLE,95750
2086531,10074905,LA CORNOUILLE,95750
